---
## STEP 7 — Pixel-Based Anomaly Segmentation Baselines
### ERFNet + Post-Hoc Methods: MSP · MaxLogit · MaxEntropy


In [ ]:
## SETUP ##

import os, sys

DRIVE_BASE   = "/content/drive/MyDrive/Fundamentals_Progetto/Coding_Part_Project"
DATASETS_DIR = os.path.join(DRIVE_BASE, "Anomaly_Validation_Datasets")
REPO_DIR     = "/content/project"          
EVAL_DIR     = os.path.join(REPO_DIR, "eval")   

if EVAL_DIR not in sys.path:
    sys.path.insert(0, EVAL_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if os.path.exists(EVAL_DIR):
    for f in sorted(os.listdir(EVAL_DIR)):
        print(f"  {f}")
else:
    print(f" Did not found eval Folder: {EVAL_DIR}")

print("Dataset available")
if os.path.exists(DATASETS_DIR):
    for d in sorted(os.listdir(DATASETS_DIR)):
        full = os.path.join(DATASETS_DIR, d)
        if os.path.isdir(full):
            n = len([
                f for f in os.listdir(full)
                if os.path.isfile(os.path.join(full, f))
            ])
            print(f"  {d}/   ({n} file top-level)")
else:
    print(f" Dataset dir not founded: {DATASETS_DIR}")


### B) Loading ERFNet pre-trained

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from erfnet import ERFNet

ERFNET_WEIGHTS = os.path.join(EVAL_DIR, "trained_models", "erfnet_pretrained.pth")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 19

erfnet_model = ERFNet(NUM_CLASSES).to(device)

state = torch.load(ERFNET_WEIGHTS, map_location=device, weights_only=True)
if "state_dict" in state:
    state = state["state_dict"]
state = {k.replace("module.", ""): v for k, v in state.items()}
erfnet_model.load_state_dict(state, strict=False)
erfnet_model.eval()

with torch.no_grad():
    dummy = torch.randn(1, 3, 512, 1024).to(device)
    out = erfnet_model(dummy)
print(f"✓ Forward pass OK — output shape: {out.shape}")

### C) Anomaly segmentation datasets 

In [ ]:
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class AnomalyDataset(Dataset):
    IMG_DIRS  = ["images", "imgs", "rgb", "JPEGImages"]
    MASK_DIRS = ["labels_masks", "masks", "labels", "annotations"]
    IMG_EXTS  = [".png", ".jpg", ".jpeg", ".PNG", ".JPG"]
    MASK_EXTS = [".png", ".jpg", ".PNG"]

    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform or IMG_TRANSFORM
        self.img_paths, self.mask_paths = self._find_pairs(root)
        assert len(self.img_paths) > 0, f"No images found in {root}."

    def _find_pairs(self, root):
        img_dir = mask_dir = None
        for d in self.IMG_DIRS:
            if os.path.isdir(os.path.join(root, d)):
                img_dir = os.path.join(root, d)
                break
        for d in self.MASK_DIRS:
            if os.path.isdir(os.path.join(root, d)):
                mask_dir = os.path.join(root, d)
                break
        if img_dir is None:
            img_dir = root

        imgs = sorted(set(
            f for ext in self.IMG_EXTS
            for f in glob.glob(os.path.join(img_dir, f"*{ext}"))
        ))

        img_paths, mask_paths = [], []
        for img_path in imgs:
            stem = os.path.splitext(os.path.basename(img_path))[0]
            if mask_dir is not None:
                for ext in self.MASK_EXTS:
                    candidate = os.path.join(mask_dir, stem + ext)
                    if os.path.exists(candidate):
                        img_paths.append(img_path)
                        mask_paths.append(candidate)
                        break
        return img_paths, mask_paths

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.img_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx])
        return self.transform(img), torch.from_numpy(np.array(mask, dtype=np.int64)), self.img_paths[idx]

#Dataset names
DATASET_REGISTRY = {
    "SMIYC RA-21":  "RoadAnomaly21",
    "SMIYC RO-21":  "RoadObstacle21",
    "FS L&F":       "FS_LostFound_full",
    "FS Static":    "fs_static",
    "Road Anomaly": "RoadAnomaly",
}

LOADERS = {}
for display_name, folder_name in DATASET_REGISTRY.items():
    path = os.path.join(DATASETS_DIR, folder_name)
    if os.path.isdir(path):
        try:
            ds = AnomalyDataset(path)
            LOADERS[display_name] = DataLoader(ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
            print(f"  {display_name:15s}  {len(ds):4d} images")
        except AssertionError as e:
            print(f"  {display_name:15s}  {e}")
    else:
        print(f" {display_name:15s}  folder not found: {folder_name}")

### D) Post-hoc scoring functions: MSP · MaxLogit · MaxEntropy

In [ ]:

def score_msp(logits: torch.Tensor) -> torch.Tensor:
    """
    Maximum Softmax Probability (MSP) .

    score = 1 − max_k softmax(z)_k   ∈ [0, 1]
    """
    probs = torch.softmax(logits, dim=1)          # [B, C, H, W]
    max_prob = probs.max(dim=1).values             # [B, H, W]
    return 1.0 - max_prob


def score_maxlogit(logits: torch.Tensor) -> torch.Tensor:
    """
    MaxLogit 

    score = −max_k z_k 
    """
    max_logit = logits.max(dim=1).values           # [B, H, W]
    return -max_logit


def score_maxentropy(logits: torch.Tensor) -> torch.Tensor:
    """
    Max Entropy (Shannon Entropy) 


    score = H(p) = −Σ_k p_k · log(p_k)   ∈ [0, log(C)]

    Normalized in [0, 1] dividing by log(C).
    """
    probs = torch.softmax(logits, dim=1)           # [B, C, H, W]
    eps = 1e-8
    entropy = -(probs * torch.log(probs + eps)).sum(dim=1)  # [B, H, W]
    
    # Normalize for max entropy (uniform distribution)
    C = logits.shape[1]
    entropy = entropy / np.log(C)                  # ∈ [0, 1]
    return entropy


SCORING_METHODS = {
    "MSP":         score_msp,
    "MaxLogit":    score_maxlogit,
    "MaxEntropy":  score_maxentropy,
}

### E) Metrics: AuPRC e FPR95

In [ ]:
from sklearn.metrics import average_precision_score, roc_curve

VOID_LABEL = 255

def compute_auprc_fpr95(scores_flat, labels_flat):
    auprc = average_precision_score(labels_flat, scores_flat)
    fpr, tpr, _ = roc_curve(labels_flat, scores_flat)
    idx = np.searchsorted(tpr, 0.95)
    fpr95 = float(fpr[min(idx, len(fpr) - 1)])
    return auprc, fpr95


def evaluate_method_on_dataset(model, score_fn, loader, dataset_name, method_name, device):
    from tqdm import tqdm
    all_scores, all_labels = [], []

    model.eval()
    with torch.no_grad():
        for imgs, masks, _ in tqdm(loader, desc=f"{dataset_name:15s} | {method_name:12s}", leave=False):
            imgs = imgs.to(device)
            masks = masks.numpy()

            logits = model(imgs)
            if not isinstance(logits, torch.Tensor):
                logits = logits[0]

            H_in, W_in = masks.shape[1], masks.shape[2]
            if logits.shape[2] != H_in or logits.shape[3] != W_in:
                logits = F.interpolate(logits, size=(H_in, W_in), mode="bilinear", align_corners=False)

            scores = score_fn(logits).squeeze(0).cpu().numpy()
            mask = masks.squeeze(0)

            valid = mask != VOID_LABEL
            if valid.sum() == 0:
                continue

            all_scores.append(scores[valid].astype(np.float32))
            all_labels.append(mask[valid].astype(np.int32))

    if not all_scores:
        return {"auprc": float("nan"), "fpr95": float("nan")}

    auprc, fpr95 = compute_auprc_fpr95(np.concatenate(all_scores), np.concatenate(all_labels))
    return {"auprc": auprc, "fpr95": fpr95}

### F) Validation ERFNet over all datsets

In [ ]:

import time

results = {ds: {} for ds in LOADERS}

total_start = time.time()

for method_name, score_fn in SCORING_METHODS.items():
    print(f"\n{'='*60}")
    print(f"  Metodo: {method_name}")
    print(f"{'='*60}")

    for ds_name, loader in LOADERS.items():
        t0 = time.time()
        res = evaluate_method_on_dataset(
            model=erfnet_model,
            score_fn=score_fn,
            loader=loader,
            dataset_name=ds_name,
            method_name=method_name,
            device=device,
        )
        results[ds_name][method_name] = res
        elapsed = time.time() - t0
        print(
            f"  {ds_name:15s}  "
            f"AuPRC={res['auprc']*100:6.2f}%  "
            f"FPR95={res['fpr95']*100:6.2f}%  "
            f"({elapsed:.0f}s)"
        )

total_elapsed = time.time() - total_start
print(f"\n Valutation completated in {total_elapsed/60:.1f} min")

### G) Save Logits

In [ ]:

LOGITS_SAVE_DIR = os.path.join(DRIVE_BASE, "saved_logits", "ERFNet")
os.makedirs(LOGITS_SAVE_DIR, exist_ok=True)

def save_erfnet_logits(model, loader, dataset_name, device, save_dir):
    from tqdm import tqdm
    ds_save_dir = os.path.join(save_dir, dataset_name.replace(" ", "_"))
    os.makedirs(ds_save_dir, exist_ok=True)

    model.eval()
    saved = 0
    with torch.no_grad():
        for imgs, masks, img_paths in tqdm(loader, desc=f"Saving logits in {dataset_name}"):
            imgs = imgs.to(device)
            output = model(imgs)
            logits = output if isinstance(output, torch.Tensor) else output[0]

            H_in, W_in = masks.shape[1], masks.shape[2]
            if logits.shape[2] != H_in or logits.shape[3] != W_in:
                logits = F.interpolate(
                    logits, size=(H_in, W_in), mode="bilinear", align_corners=False
                )

            stem = os.path.splitext(os.path.basename(img_paths[0]))[0]
            torch.save(
                {"logits": logits.squeeze(0).half().cpu(),
                 "mask":   masks.squeeze(0)},
                os.path.join(ds_save_dir, f"{stem}.pt")
            )
            saved += 1
    print(f" {saved} File saved in {ds_save_dir}")


def eval_from_saved_logits(save_dir, dataset_name, score_fn):
    ds_dir = os.path.join(save_dir, dataset_name.replace(" ", "_"))
    all_scores, all_labels = [], []

    for pt_file in sorted(glob.glob(os.path.join(ds_dir, "*.pt"))):
        data = torch.load(pt_file, weights_only=True)
        logits = data["logits"].float().unsqueeze(0)   # [1, C, H, W]
        mask   = data["mask"].numpy()                  # [H, W]

        scores = score_fn(logits).squeeze(0).numpy()   # [H, W]
        valid  = mask != VOID_LABEL
        if valid.sum() > 0:
            all_scores.append(scores[valid].astype(np.float32))
            all_labels.append(mask[valid].astype(np.int32))

    all_scores = np.concatenate(all_scores)
    all_labels = np.concatenate(all_labels)
    auprc, fpr95 = compute_auprc_fpr95(all_scores, all_labels)
    return {"auprc": auprc, "fpr95": fpr95}


# Uncomment to save logit on Drive only one time on the 
# next session we need to use eval_from_saved_logits --> PRO TIPPP :)

# for ds_name, loader in LOADERS.items():
#     save_erfnet_logits(erfnet_model, loader, ds_name, device, LOGITS_SAVE_DIR)

print(f" Function save/load logits ready")
print(f"  Save dir: {LOGITS_SAVE_DIR}")